In [ ]:
import os
import shutil

# 1. Dọn dẹp rác gây xung đột (Quan trọng)
if os.path.exists("speechmos"): 
    print("🧹 Đang xóa folder speechmos rác...")
    shutil.rmtree("speechmos")

# 2. Cài đặt lại thư viện sạch
!pip uninstall -y speechmos torchaudio torch # Gỡ bản cũ
!pip install -q openai-whisper jiwer librosa fastdtw scipy numpy pandas textgrid
# Cài torch bản ổn định nhất cho Kaggle
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu118

print("✅ Môi trường đã sạch sẽ!")

In [ ]:
import os
import glob
import torch
import torchaudio
import librosa
import numpy as np
import whisper
import jiwer
import pandas as pd
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw
from scipy.interpolate import interp1d
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings("ignore")

# ================= CẤU HÌNH ĐƯỜNG DẪN =================
GEN_AUDIO_FOLDER = "/kaggle/input/score2/ket_qua_test_full_pack_1/gen" 
REF_AUDIO_FOLDER = "/kaggle/input/score2/ket_qua_test_full_pack_1/ref" 
TEST_FILE_LIST   = "/kaggle/input/score2/ket_qua_test_full_pack_1/text.txt"
# ======================================================

class FinalEvaluator:
    def __init__(self, device="cuda"):
        self.device = device
        print(f"🖥️ Đang khởi tạo trên {device}...")
        
        # 1. LOAD UTMOS
        print("⏳ Đang tải UTMOS...")
        try:
            self.mos_predictor = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True)
            self.mos_predictor = self.mos_predictor.to(device)
            self.mos_predictor.eval()
            print("✅ UTMOS OK!")
        except:
            self.mos_predictor = None
        
        # 2. LOAD WHISPER
        print("⏳ Đang tải Whisper...")
        self.asr_model = whisper.load_model("medium", device=device)

    def calculate_mos(self, wav_path):
        if self.mos_predictor is None: return 0
        try:
            wave, sr = torchaudio.load(wav_path)
            if sr != 16000:
                resampler = torchaudio.transforms.Resample(sr, 16000)
                wave = resampler(wave)
            if wave.shape[0] > 1: wave = wave.mean(dim=0, keepdim=True)
            wave = wave.to(self.device)
            with torch.no_grad(): score = self.mos_predictor(wave, 16000)
            return score.item()
        except: return 0

    def trim_f0(self, f0):
        """Hàm cắt bỏ các giá trị 0 (không có cao độ) ở đầu và đuôi mảng"""
        # Tìm vị trí đầu tiên và cuối cùng khác 0
        non_zeros = np.where(f0 > 0)[0]
        if len(non_zeros) == 0:
            return np.array([])
        start, end = non_zeros[0], non_zeros[-1]
        return f0[start:end+1]

    def resize_array(self, arr, target_len):
        """Co giãn mảng arr để có độ dài bằng target_len"""
        if len(arr) == target_len: return arr
        x_old = np.linspace(0, 1, len(arr))
        x_new = np.linspace(0, 1, target_len)
        f = interp1d(x_old, arr, kind='linear')
        return f(x_new)

    def calculate_f0(self, gen_path, ref_path):
        try:
            # 1. Load file (Mono + Trim Silence Audio)
            y_gen, sr = librosa.load(gen_path, sr=22050, mono=True)
            y_ref, sr = librosa.load(ref_path, sr=22050, mono=True)
            
            # Cắt khoảng lặng audio
            y_gen, _ = librosa.effects.trim(y_gen, top_db=20)
            y_ref, _ = librosa.effects.trim(y_ref, top_db=20)

            # Ép 1 chiều
            y_gen = y_gen.flatten()
            y_ref = y_ref.flatten()

            # Nếu file quá ngắn sau khi trim -> Bỏ
            if len(y_gen) < 512 or len(y_ref) < 512: return None, None
            
            # 2. Trích xuất F0
            f0_gen, _, _ = librosa.pyin(y_gen, fmin=60, fmax=1100, sr=sr, frame_length=2048)
            f0_ref, _, _ = librosa.pyin(y_ref, fmin=60, fmax=1100, sr=sr, frame_length=2048)
            
            f0_gen = np.nan_to_num(f0_gen)
            f0_ref = np.nan_to_num(f0_ref)

            # 3. Trim F0 (Cắt bỏ đoạn F0 = 0 ở đầu đuôi)
            f0_gen = self.trim_f0(f0_gen)
            f0_ref = self.trim_f0(f0_ref)

            if len(f0_gen) < 10 or len(f0_ref) < 10: return None, None

            # 4. CHIẾN THUẬT 1: Thử DTW trước (Chính xác nhất)
            try:
                distance, path = fastdtw(f0_gen, f0_ref, dist=euclidean)
                gen_v = np.array([f0_gen[i] for i, j in path])
                ref_v = np.array([f0_ref[j] for i, j in path])
                
                # Tính trên đoạn voiced
                idx = (gen_v > 0) & (ref_v > 0)
                if np.sum(idx) > 10:
                    rmse = np.sqrt(np.mean((gen_v[idx] - ref_v[idx]) ** 2))
                    corr = np.corrcoef(gen_v[idx], ref_v[idx])[0, 1]
                    return rmse, corr
            except:
                pass # Nếu DTW lỗi, chuyển sang chiến thuật 2

            # 5. CHIẾN THUẬT 2: Forced Interpolation (Ép độ dài bằng nhau)
            # Resize mảng ngắn theo mảng dài
            target_len = max(len(f0_gen), len(f0_ref))
            f0_gen_res = self.resize_array(f0_gen, target_len)
            f0_ref_res = self.resize_array(f0_ref, target_len)

            rmse = np.sqrt(np.mean((f0_gen_res - f0_ref_res) ** 2))
            corr = np.corrcoef(f0_gen_res, f0_ref_res)[0, 1]
            
            return rmse, corr

        except Exception as e:
            # print(f"Lỗi: {e}")
            return None, None

    def run(self):
        ground_truth = {}
        if os.path.exists(TEST_FILE_LIST):
            with open(TEST_FILE_LIST, 'r', encoding='utf-8') as f:
                for line in f:
                    p = line.strip().split('|')
                    if len(p) >= 2: ground_truth[os.path.basename(p[0]).replace(".wav", "")] = p[1]

        files = glob.glob(os.path.join(GEN_AUDIO_FOLDER, "*.wav"))
        print(f"🚀 Bắt đầu chấm {len(files)} file...")

        results = []
        for path in tqdm(files):
            fname = os.path.basename(path).replace(".wav", "")
            item = {"Filename": fname}
            
            # 1. MOS
            item["MOS"] = self.calculate_mos(path)
            
            # 2. WER/CER
            if fname in ground_truth:
                ref_text = ground_truth[fname].lower().replace(".", "").strip()
                hyp_text = self.asr_model.transcribe(path, language="vi")["text"].lower().replace(".", "").strip()
                item["WER"] = jiwer.wer(ref_text, hyp_text)
                item["CER"] = jiwer.cer(ref_text, hyp_text)
            else:
                item["WER"], item["CER"] = np.nan, np.nan
            
            # 3. F0
            ref_path = os.path.join(REF_AUDIO_FOLDER, fname + ".wav")
            if os.path.exists(ref_path):
                item["F0_RMSE"], item["F0_Corr"] = self.calculate_f0(path, ref_path)
            else:
                item["F0_RMSE"], item["F0_Corr"] = np.nan, np.nan
                
            results.append(item)
            
        return pd.DataFrame(results)

if __name__ == "__main__":
    if os.path.exists(GEN_AUDIO_FOLDER):
        ev = FinalEvaluator()
        df = ev.run()
        
        print("\n" + "="*40)
        print(f"⭐ UTMOS:   {df['MOS'].mean():.4f}")
        print(f"📝 CER:     {df['CER'].mean()*100:.2f}%")
        print(f"📝 WER:     {df['WER'].mean()*100:.2f}%")
        
        f0_valid = df.dropna(subset=['F0_RMSE'])
        if len(f0_valid) > 0:
            print(f"📈 F0 RMSE: {f0_valid['F0_RMSE'].mean():.2f}")
            print(f"📈 F0 Corr: {f0_valid['F0_Corr'].mean():.4f}")
            print(f"✅ Đã đo được F0: {len(f0_valid)}/{len(df)} file")
        else:
            print("❌ Vẫn không đo được. (File gốc và Gen quá khác biệt).")
            
        print("="*40)
        df.to_csv("final_score.csv", index=False)
        print("✅ Đã lưu file kết quả!")
    else:
        print(f"❌ Không tìm thấy: {GEN_AUDIO_FOLDER}")